# Lab 00 — 從網卡 counter 到告警的 pipeline

作業系統的 counter 進 node_exporter，Prometheus 每 5 秒抓一次。一支 Python 服務回頭查 Prometheus，
算出偏離分數，把分數曝露成 `/metrics`，於是 Prometheus 把分數也抓回去。Grafana 與告警規則查的就是
那個分數。

![Lab 00 的資料流](../../diagrams/lab00_pipeline.svg)

圖上五個方框，這門課只寫橘色那一個。pipeline 接好之後沒有人需要按執行。

這一節照訊號流動的順序走。先看原始 counter 是什麼形狀，動手算一次分數，把同一段計算變成常駐服務，
確認分數回到 Prometheus，最後畫成 panel 並讓告警響一次。

Lab 01 與 Lab 02 換掉的只有那支服務裡算分數的函式。pipeline 本身接完就不再變動。

## 1. 前置的三個服務

| 元件 | 位址 | 這一節的用途 |
| --- | --- | --- |
| Prometheus | <http://localhost:9090> | 存指標、被 detector 查詢、評估告警規則 |
| node_exporter | <http://localhost:9100/metrics> | 曝露這台機器的網路 counter |
| Grafana | <http://localhost:3000> | 把原始速率與分數畫在同一張圖上 |

安裝與啟動見 [`labs/getting-started/`](../getting-started/README.md)。Windows 的 exporter 換成
`windows_exporter`，聽在 9182。第四個服務到第 5 節才啟動。

In [ ]:
import pathlib
import sys
import time
import urllib.request
from collections import deque

import matplotlib.pyplot as plt
import pandas as pd
import requests

PROJECT_ROOT = pathlib.Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "environments").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
LAB_DIR = PROJECT_ROOT / "labs" / "workshop"

PROM = "http://localhost:9090"

plt.rcParams.update({
    "font.family": "serif", "font.serif": ["Georgia", "Times New Roman", "DejaVu Serif"],
    "figure.dpi": 110, "axes.titlesize": 11, "axes.titlelocation": "left",
    "axes.labelsize": 9, "legend.fontsize": 8, "xtick.labelsize": 8, "ytick.labelsize": 8,
    "axes.grid": True, "grid.alpha": 0.22, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
})


def reachable(url, timeout=2.0):
    """能不能拿到 HTTP 回應。用 stdlib，這一格在任何環境都要跑得起來。"""
    try:
        with urllib.request.urlopen(url, timeout=timeout) as response:
            return response.status < 400
    except Exception:
        return False


SERVICES = [
    ("Prometheus", f"{PROM}/-/ready", "第 2 節起每一格都要用"),
    ("node_exporter", "http://localhost:9100/metrics", "Windows 改查 9182"),
    ("Grafana", "http://localhost:3000/api/health", "第 7 節才用得到"),
]

print(f"專案根目錄  {PROJECT_ROOT}\n")
for name, url, note in SERVICES:
    print(f"  {'OK    ' if reachable(url) else '沒回應'}  {name:<16}{url:<42}{note}")

## 2. scrape 狀態與 `up`

`up` 由 Prometheus 為每一個 target 自己產生，抓得到是 1，抓不到是 0。空白的畫面該查的第一條 PromQL
永遠是它，因為只有它分得開「沒有資料」與「這段時間沒有異常」。

`aiops-detector` 現在會是 0，或者整個 job 不存在。它抓的是第 5 節才啟動的那支服務。

In [ ]:
def promql(query):
    """打 Prometheus 的 instant query API。Grafana 在背後做的是同一件事。"""
    response = requests.get(f"{PROM}/api/v1/query", params={"query": query}, timeout=10)
    response.raise_for_status()
    return response.json()["data"]["result"]


targets = requests.get(f"{PROM}/api/v1/targets", timeout=5).json()["data"]["activeTargets"]
print(f"{'job':<18}{'endpoint':<40}{'health'}")
for t in targets:
    print(f"  {t['labels']['job']:<18}{t['scrapeUrl']:<40}{t['health']}")

print(f"\n{'job':<18}{'up'}")
for row in promql("up"):
    print(f"  {row['metric']['job']:<18}{row['value'][1]}")

## 3. counter 與 rate

`node_network_receive_bytes_total` 是 counter，從開機起只增不減，直到重開機歸零。直接畫它得到一條
往上的斜線。有意義的是斜率，也就是每秒多少 bytes，PromQL 用 `rate()` 取斜率。

```promql
node_network_receive_bytes_total{device="en0"}             # 累積量
rate(node_network_receive_bytes_total{device="en0"}[1m])   # 每秒速率
```

網卡要挑對。虛擬介面很多，挑錯的話後面每一張圖都是一條平的零線，所以下面這一格用 `topk` 讓資料自己
指出哪一張在傳。

In [ ]:
# node_exporter 與 windows_exporter 量同一件事，名字與 label 不同，兩個都問一次。
EXPORTERS = [
    ("node_network_receive_bytes_total", "device"),
    ("windows_net_bytes_received_total", "nic"),
]

METRIC = LABEL = IFACE = None
for metric, label in EXPORTERS:
    rows = promql(f"topk(3, rate({metric}[1m]))")
    if rows:
        METRIC, LABEL = metric, label
        print(f"最近一分鐘接收速率前三名（{metric}）")
        for row in rows:
            print(f"  {row['metric'][label]:<14}{float(row['value'][1]):>14,.0f} bytes/s")
        IFACE = rows[0]["metric"][label]
        break

assert IFACE, "查不到任何網卡指標，先確認 exporter 起來了、而且被 Prometheus 抓到"
print(f"\n後面用 IFACE = {IFACE!r}")

selector = f'{METRIC}{{{LABEL}="{IFACE}"}}'
print("\ncounter 與 rate，同一個瞬間")
print(f"  累積量  {float(promql(selector)[0]['value'][1]):>18,.0f} bytes    開機以來的總和")
print(f"  rate    {float(promql(f'rate({selector}[1m])')[0]['value'][1]):>18,.0f} bytes/s  這才是流量")

`query_range` 是 Grafana 畫折線圖時打的端點。下面這張圖與待會 dashboard 第一張 panel 讀的是同一批
數字，兩邊對不起來就表示中間有一段斷了。

In [ ]:
def promql_range(query, minutes=60, step="15s"):
    """拉一段時間序列。查不到資料時回 None，不要讓 notebook 中斷。"""
    end = time.time()
    response = requests.get(f"{PROM}/api/v1/query_range",
                            params={"query": query, "start": end - minutes * 60,
                                    "end": end, "step": step}, timeout=20)
    response.raise_for_status()
    result = response.json()["data"]["result"]
    if not result:
        print("這段時間沒有資料。Prometheus 剛啟動的話，等幾分鐘再執行一次。")
        return None
    pairs = result[0]["values"]
    return pd.DataFrame({
        "timestamp": pd.to_datetime([float(t) for t, _ in pairs], unit="s"),
        "rate_bps": [float(v) for _, v in pairs],
    })


history = promql_range(f"rate({selector}[1m])")
print(f"{len(history)} 個樣本   {history['timestamp'].min()} 到 {history['timestamp'].max()}")
print(history["rate_bps"].describe().round(0).to_string())

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(history["timestamp"], history["rate_bps"], color="#3B7DD8", lw=1.2)
ax.set_title(f"receive rate on {IFACE}, last hour", loc="left")
ax.set_ylabel("bytes/s")
fig.tight_layout()
plt.show()

## 4. 滾動 z 分數

眼睛看得出哪一段不尋常，規則引擎不行，它需要一個數字。最素樸的做法是問每一個點離前面那段時間的常態
有多遠，用標準差當單位，這個量叫 z 分數。

有一個細節決定它成不成立。分數必須用「這個值進入視窗之前」的視窗算，反過來的話，異常值會先被算進
平均與標準差裡，自己抬高自己的基線，於是越大的異常越不容易被抓到。

函式寫在 [`detector.py`](detector.py)，下面直接 import 進來。第 5 節那支常駐服務跑的是同一段程式碼。

In [ ]:
sys.path.insert(0, str(LAB_DIR))
from detector import MIN_SAMPLES, WINDOW, rolling_zscore  # noqa: E402

import inspect  # noqa: E402
print(inspect.getsource(rolling_zscore))
print(f"WINDOW = {WINDOW} 個樣本，MIN_SAMPLES = {MIN_SAMPLES}")


def score_series(values, window_len=WINDOW):
    """照 detector.py 主迴圈的順序評分：先用舊視窗算，再把新值放進視窗。"""
    window, scores = deque(maxlen=window_len), []
    for value in values:
        scores.append(rolling_zscore(window, value))
        window.append(value)
    return scores


history["score"] = score_series(history["rate_bps"])
print(f"\n分數絕對值的 95 百分位  {history['score'].abs().quantile(0.95):.2f}")
print(f"超過門檻 3 的樣本        {int((history['score'].abs() > 3).sum())} / {len(history)}")

fig, (top, bottom) = plt.subplots(2, 1, figsize=(12, 5), sharex=True,
                                  gridspec_kw={"height_ratios": [2, 1]})
top.plot(history["timestamp"], history["rate_bps"], color="#3B7DD8", lw=1.2)
top.set_title(f"receive rate on {IFACE}", loc="left")
top.set_ylabel("bytes/s")
bottom.plot(history["timestamp"], history["score"], color="#E0752D", lw=1.2)
for edge in (3, -3):
    bottom.axhline(edge, color="#D6455D", ls="--", lw=1.0)
bottom.set_title("rolling z-score, threshold at 3", loc="left")
bottom.set_ylabel("z")
fig.tight_layout()
plt.show()

視窗長度是這個偵測器唯一的旋鈕，兩端都會出事。下面把同一段資料用三種視窗各算一次。先猜哪一種越線的
樣本最多，再執行。

In [ ]:
rows = []
fig, ax = plt.subplots(figsize=(12, 3.2))
for window_len, color in [(10, "#98A2B3"), (WINDOW, "#E0752D"), (160, "#3B7DD8")]:
    scores = pd.Series(score_series(history["rate_bps"], window_len))
    rows.append({
        "window_samples": window_len,
        "window_minutes": round(window_len * 15 / 60, 1),
        "p95_abs_score": round(scores.abs().quantile(0.95), 2),
        "over_threshold": int((scores.abs() > 3).sum()),
    })
    ax.plot(history["timestamp"], scores, lw=1.1, color=color, label=f"window={window_len}")

ax.axhline(3, color="#D6455D", ls="--", lw=1.0)
ax.axhline(-3, color="#D6455D", ls="--", lw=1.0)
ax.set_title("same traffic, three window lengths", loc="left")
ax.set_ylabel("z")
ax.legend(ncol=3)
fig.tight_layout()
plt.show()

print(pd.DataFrame(rows).to_string(index=False))

短視窗把日常起伏也算成異常，長視窗讓慢慢爬上去的異常被基線跟著吃掉。這台機器的即時流量沒有真值可以
裁決哪一個對。Lab 01 換成標好事件的歷史資料，用 event recall 把這個取捨量出來。

## 5. 常駐的偵測服務

上面幾格算完就結束了。同一段計算要持續運作，需要的只是一個迴圈，加上一個讓 Prometheus 抓得到的 HTTP
端點。`prometheus_client` 這個官方套件把後者變成一行 `start_http_server`。

`detector.py` 就是這樣一支程式。另外開一個終端機，在 repo 根目錄執行下面這一行，視窗留著不要關。

```bash
python labs/workshop/detector.py
```

終端機印出 `detector 監看 ...` 之後，回來執行下一格。

In [ ]:
EXPORT_PORT = 9200

if not reachable(f"http://localhost:{EXPORT_PORT}/metrics"):
    print(f":{EXPORT_PORT} 沒有回應。先在另一個終端機執行 python labs/workshop/detector.py，再跑這一格。")
else:
    metrics = requests.get(f"http://localhost:{EXPORT_PORT}/metrics", timeout=5).text
    print(f"detector 在 :{EXPORT_PORT}/metrics 曝露的內容\n")
    for line in metrics.splitlines():
        if line.startswith("aiops_") and not line.startswith("#"):
            print(f"  {line}")
    print("\n對照 node_exporter 的同一批欄位")
    node = requests.get("http://localhost:9100/metrics", timeout=5).text
    for line in node.splitlines():
        if line.startswith("node_network_receive_bytes_total") and IFACE in line:
            print(f"  {line}")
            break

兩邊印出來的行長得一模一樣，指標名、label、值。Prometheus 認得的只有這三項。自己寫的服務與官方
exporter 在這一層沒有分別，pipeline 接得起來靠的就是這一點。

剛啟動的時候分數是 0，樣本數也還很少，那段是暖機期。兩三個樣本算出來的標準差小到沒有意義，隨便一點
變動都換算成十幾二十的分數，所以 `rolling_zscore` 在視窗裝滿 `MIN_SAMPLES` 之前一律回 0。

## 6. 分數回到 Prometheus

`aiops-detector` 這個 job 已經寫在三份 `infra/prometheus/prometheus.*.yml` 裡，指向
`localhost:9200`。Prometheus 比 detector 早啟動的話，讓它重讀一次設定。

```bash
curl -X POST http://localhost:9090/-/reload
```

Windows PowerShell 用 `Invoke-WebRequest -Method Post http://localhost:9090/-/reload`。回 `405` 表示
啟動時沒有帶 `--web.enable-lifecycle`，重啟 Prometheus 也可以。

In [ ]:
print(f"{'job':<18}{'up'}")
for row in promql("up"):
    print(f"  {row['metric']['job']:<18}{row['value'][1]}")

print("\n分數從 Prometheus 這一側查回來")
for row in promql("aiops_traffic_score"):
    print(f"  {LABEL}={row['metric'].get('device', '?'):<10}{float(row['value'][1]):>8.2f}")

# 繞回來之後，PromQL 對分數做得了跟對任何指標一樣的事。
print("\n拿 PromQL 直接處理這個分數")
for expr, note in [
    ("max_over_time(abs(aiops_traffic_score)[10m:15s])", "近 10 分鐘的峰值，subquery"),
    ("avg_over_time(aiops_traffic_bps[10m:15s])", "近 10 分鐘的平均速率"),
    ("aiops_detector_window_samples", "視窗裝了幾個樣本"),
]:
    rows = promql(expr)
    value = f"{float(rows[0]['value'][1]):.2f}" if rows else "沒有資料"
    print(f"  {value:>10}   {note}")

`up{job="aiops-detector"}` 是 1，`aiops_traffic_score` 也查得回來，pipeline 就閉合了。

分數送回 Prometheus 換到的是上面那三條查詢。subquery、區間平均、任何一種 PromQL 的工具，對它一律
有效。分數留在 notebook 裡的話，這些工具一個都用不上。

## 7. Grafana 的三張 panel

到 <http://localhost:3000> 建一張 dashboard，逐步的做法寫在 [`dashboard.md`](dashboard.md)。

下面這一格先把三張 panel 要貼的 PromQL 各查一次，查得回來再去建 panel。查詢回空集合的時候，panel
的編輯畫面不會顯示任何錯誤訊息，只給你一張空白的圖，所以除錯不要在那裡做。

In [ ]:
PANELS = [
    ("Throughput, receive and transmit", f'rate({selector}[1m])'),
    ("Anomaly score", "aiops_traffic_score"),
    ("Alert state", 'ALERTS{alertname="TrafficAnomaly"}'),
]

for title, expr in PANELS:
    rows = promql(expr)
    status = f"{len(rows)} series" if rows else "空的"
    print(f"  {status:<12}{title:<38}{expr}")

print("\nAlert state 現在是空的屬於正常，第 8 節會讓它有東西。")

## 8. 告警規則與 `for`

`infra/prometheus/alerts.yml` 裡的 `TrafficAnomaly` 打在 `aiops_traffic_score` 上，門檻 3，`for: 1m`。

`for` 是這條規則裡最該理解的一個字。條件成立的瞬間告警不會送出去，規則先進入 Pending，連續成立滿
60 秒才轉成 Firing。這是時間閘，擋掉的是越線的單一取樣。Lab 02 在 Python 那一側把同一件事寫成
「連續 n 個樣本才算數」。

先看規則檔現在載進去了什麼。

In [ ]:
def rule_table():
    groups = requests.get(f"{PROM}/api/v1/rules", timeout=5).json()["data"]["groups"]
    return [r for g in groups for r in g["rules"]]


def alert_state(name):
    for rule in rule_table():
        if rule.get("name") == name:
            return rule.get("state", "-")
    return "-"


print(f"{'規則':<24}{'型別':<12}{'狀態':<12}{'for'}")
for rule in rule_table():
    hold = f"{rule['duration']:.0f}s" if rule.get("duration") else "-"
    print(f"  {rule.get('name', '?'):<24}{rule['type']:<12}{rule.get('state', '-'):<12}{hold}")

下面這一格開四條連線抓資料，每 10 秒問一次分數與告警狀態，直到 `TrafficAnomaly` 轉成 Firing 或者
等滿時限。執行之前先寫下兩個猜測，分數要幾秒越過 3，越過之後又要幾秒轉 Firing。

這一格會下載幾百 MB。用行動網路的話跳過它，直接讀下一格的說明。

In [ ]:
import threading  # noqa: E402

LOAD_URL = "https://speed.cloudflare.com/__down?bytes=800000000"
STREAMS, POLL_SECONDS, DEADLINE = 4, 10, 240


def pull_traffic():
    try:
        with requests.get(LOAD_URL, stream=True, timeout=DEADLINE) as response:
            for _ in response.iter_content(chunk_size=1 << 20):
                pass
    except Exception:
        pass


for _ in range(STREAMS):
    threading.Thread(target=pull_traffic, daemon=True).start()

started, peak, crossed_at = time.time(), 0.0, None
print(f"{'秒':>5}{'rate bytes/s':>18}{'score':>10}   TrafficAnomaly")
while time.time() - started < DEADLINE:
    elapsed = time.time() - started
    rate = promql("aiops_traffic_bps")
    score = promql("aiops_traffic_score")
    value = float(score[0]["value"][1]) if score else 0.0
    state = alert_state("TrafficAnomaly")
    peak = max(peak, abs(value))
    if abs(value) > 3 and crossed_at is None:
        crossed_at = elapsed
    print(f"{elapsed:>5.0f}{float(rate[0]['value'][1]) if rate else 0:>18,.0f}{value:>10.2f}   {state}")
    if state == "firing":
        print(f"\nFiring。分數在第 {crossed_at:.0f} 秒越過 3，再過 {elapsed - crossed_at:.0f} 秒轉 Firing，"
              f"後面那段就是 for: 1m。")
        break
    time.sleep(POLL_SECONDS)
else:
    print(f"\n分數峰值 {peak:.2f}，沒有越過門檻 3。下一格的說明解釋為什麼這是預期內的結果。")

兩種結果都要能解釋。

轉成 Firing 的話，表裡有兩段延遲。分數越過 3 之前的那幾行，是視窗在等夠多個高於基線的樣本，長度
由演算法決定。越過 3 之後到轉 Firing 的那 60 秒由 `for` 決定。兩者相加才是真正的偵測延遲，只調
其中一個，另一個不會跟著動。

峰值停在 3 以下的話，原因通常是這個基線的演算法。視窗是 10 分鐘的滾動平均，
下載持續整段時間，於是流量一邊抬高分子，一邊把平均與標準差也一起抬高。分母跟著分子長，商就漲不
上去。要抓的那個異常，自己稀釋了用來抓它的那把尺。

這不是設定錯誤，是滾動平均的固有弱點，而且它在真實故障上最常出現的形式正是這一種，持續而非瞬間。
Lab 01 用中位數與 MAD 換掉這把尺，並且拿標好事件的資料量出換掉之後多抓到幾個。

## 9. 四種故障模式

pipeline 的接點有幾個，故障模式就有幾種。下面四種各做一次，每一次先預測畫面上會看到什麼，再動手。

1. **關掉 detector。** 在那個終端機按 `Ctrl+C`，等 60 秒。`up{job="aiops-detector"}` 變成 0，
   `DetectorDown` 開始響。分數那張 panel 斷線，而 `TrafficAnomaly` 不會響，因為它的條件再也不會
   成立。監看系統自己掛掉的時候畫面看起來是安靜的，`DetectorDown` 就是為了這件事存在。
2. **關掉 node_exporter。** detector 查不到資料，分數停在最後一個值，速率那張 panel 也斷線，
   `up{job="node-exporter"}` 變成 0。
3. **時間範圍選錯。** 把 Grafana 的時間選擇器改成 Last 5 minutes 再改成 Last 2 days。panel 變空白，
   而且沒有任何錯誤訊息。
4. **規則檔改壞。** 在 `alerts.yml` 裡把某個指標名打錯再 reload。規則載得進去，只是永遠不會成立。
   `promtool check rules infra/prometheus/alerts.yml` 檢查得出語法錯誤，檢查不出指標名打錯。

第一種做完之後執行下一格，它印的是 Prometheus 這一側看到的狀態。做完把 detector 開回來。

In [ ]:
detector_up = promql('up{job="aiops-detector"}')
print(f"up{{job=\"aiops-detector\"}}  {detector_up[0]['value'][1] if detector_up else '沒有這個 job'}")
print(f"DetectorDown            {alert_state('DetectorDown')}")
print(f"TrafficAnomaly          {alert_state('TrafficAnomaly')}")

stale = promql("aiops_traffic_score")
if stale:
    print(f"\n分數還查得到，值是 {float(stale[0]['value'][1]):.2f}。")
    print("停掉的 detector 會讓這個值停在最後一次寫入，Prometheus 抓不到之後才讓它消失，")
    print("所以「分數看起來正常」不等於「偵測還在運作」。")

第三種與第四種跟前兩種的差別在痕跡。前兩種留下 `up` 的狀態變化，後兩種什麼都不留。所以看到空白畫面
的時候順序是固定的，先查 `up`，再看時間範圍，最後才懷疑資料本身。

## 10. 自我檢核

下面五題自己回答一次。答不出來的那一題，就是這一節還沒接上的地方。

1. counter 與 rate 的差別，以及為什麼直接畫 `node_network_receive_bytes_total` 沒有意義。
2. `aiops_traffic_score` 是怎麼進到 Prometheus 的，中間經過哪些元件。答案裡不應該出現「匯入」
   或「上傳」這兩個詞。
3. `TrafficAnomaly` 這條規則怎麼知道分數是 Python 算的。它不知道，說明為什麼這件事是優點。
4. 第 8 節那張表裡的兩段延遲，各自由誰決定，想縮短總延遲該動哪一個。
5. detector 掛掉的時候，哪一條規則會響，哪一條不會，為什麼。

pipeline 到這裡就完整了。Lab 01 與 Lab 02 要處理的，是把 `rolling_zscore` 換成撐得住真實流量的版本。
下一節先問更前面的問題。同一段流量換一種基線就換一種判定，那應該換哪一種。